In [1]:
print("yo")
print("oh no")

yo
oh no


## Utils

In [2]:
!pip install imbalanced-learn


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Source - https://stackoverflow.com/a
# Posted by Afshin Amiri, modified by community. See post 'Timeline' for change history
# Retrieved 2025-11-17, License - CC BY-SA 4.0

# ---- RUN THIS TO UNZIP FOLDER ----
#import zipfile as zf
#files = zf.ZipFile("datasets.zip", 'r')
#files.extractall('./')
#files.close()


# Data

## Abalone-17_vs_7-8-9-10 dataset

In [4]:
import numpy as np
import os.path
import pandas as pd
from sklearn.preprocessing import StandardScaler

### Prepare data

In [5]:
file_name = "./datasets/abalone-17_vs_7-8-9-10/abalone-17_vs_7-8-9-10.dat"
#if os.path.exists(file_name):
#    print("exists")

df = pd.read_csv(
    file_name,
    comment='@',
    header=None,
    sep=','
)

columns = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight",
    "Viscera_weight", "Shell_weight",
    "Class"
]

df.columns = columns

df = pd.get_dummies(df, columns=["Sex"], dtype=int)

# print(df["Class"].unique())
df["Class"] = df["Class"].map({" negative": 0, " positive": 1})

X = df.drop("Class", axis=1).values
y = df["Class"].values

df

#folds = load_data(file_name, type_data="imbalanced", raw=True)

,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Class,Sex_F,Sex_I,Sex_M
0,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,0,0,0,1
1,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,0,0,1,0
2,0.355,0.280,0.085,0.2905,0.0950,0.0395,0.115,0,0,1,0
3,0.365,0.295,0.080,0.2555,0.0970,0.0430,0.100,0,0,0,1
4,0.390,0.295,0.095,0.2030,0.0875,0.0450,0.075,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
2333,0.570,0.440,0.140,0.9535,0.3785,0.2010,0.305,1,1,0,0
2334,0.585,0.455,0.125,1.0270,0.3910,0.2120,0.250,1,0,0,1
2335,0.620,0.485,0.220,1.5110,0.5095,0.2840,0.510,1,1,0,0
2336,0.635,0.505,0.185,1.3035,0.5010,0.2950,0.410,1,1,0,0


### Data cleaning

In [6]:
X = df.drop(columns=["Class", "Sex_F", "Sex_I", "Sex_M"], errors='ignore').values
y = df["Class"].values

count_class = df["Class"].value_counts()

imb_count = df["Class"].value_counts(normalize=True) * 100
imb_count

imb_ratio = count_class.max() / count_class.min()
imb_ratio

np.float64(39.310344827586206)

# Models / Pipeline definitions

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import ADASYN

from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier


In [ ]:
def create_pipeline(architecture, data_level_method=None):
    steps = []
    
    if data_level_method == "smote":
        steps.append(("smote", SMOTE(random_state=0)))
    elif data_level_method == "adasyn":
        steps.append(("adasyn", ADASYN(random_state=0)))
    
    if architecture == "mlp":
        steps.append((
            "mlp", 
            MLPClassifier(
                (5,10,5), 
                max_iter=2000, 
                solver='adam', 
                learning_rate_init=0.001)
        ))
    elif architecture == "CART":
        steps.append((
            "cart", 
            DecisionTreeClassifier(
                random_state=0)
        ))
        
    
    pipeline = ImbPipeline(steps=steps)
    return pipeline

In [9]:
pipelines_dict = {}

In [10]:
pipelines_dict["base_mlp"] = create_pipeline("mlp", data_level_method=None)
pipelines_dict["smote_mlp"] = create_pipeline("mlp", data_level_method="smote")
pipelines_dict["adasyn_mlp"] = create_pipeline("mlp", data_level_method="adasyn")


# Training and export

In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from imblearn.metrics import geometric_mean_score

def shuffle_data_and_train_and_evaluate(X, y, pipeline):
    valid_precisions = []
    valid_recalls = []

    valid_f1s = []
    valid_gmeans = []
    valid_aucs = []

    runs_target = 100
    runs_cur = 0
    seed = 0

    while runs_cur < runs_target:
        # Random Split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, 
            test_size=0.3, 
            stratify=None, 
            random_state=seed
        )
        
        n_minority = sum(y_test == 1)
        
        if n_minority >= 10:
            # Scale
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
            
            # Train
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            y_score = pipeline.predict_proba(X_test)[:, 1] # For AUC calculation
            
            valid_precisions.append(precision_score(y_test, y_pred, pos_label=1, zero_division=0))
            valid_recalls.append(recall_score(y_test, y_pred, pos_label=1, zero_division=0))
            valid_f1s.append(f1_score(y_test, y_pred, pos_label=1, zero_division=0))
            valid_gmeans.append(geometric_mean_score(y_test, y_pred, pos_label=1))
            valid_aucs.append(roc_auc_score(y_test, y_score))
            
            runs_cur += 1
            
            if runs_cur % 10 == 0:
                print(f"Run {runs_cur}/{runs_target} completed...")

        seed += 1


    print(f"Seeds checked: {seed}")
    print(f"Average Precision: {np.mean(valid_precisions):.4f}")
    print(f"Average Recall:    {np.mean(valid_recalls):.4f}")
    print("")
    print(f"Average F1:        {np.mean(valid_f1s):.4f}")
    print(f"Standard deviation F1: {np.std(valid_f1s):.4f}")
    print(f"Average G-Mean:    {np.mean(valid_gmeans):.4f}")
    print(f"standard deviation G-Mean: {np.std(valid_gmeans):.4f}")
    print(f"Average AUC:      {np.mean(valid_aucs):.4f}")
    print(f"standard deviation AUC: {np.std(valid_aucs):.4f}")
    
    output_dict = {
        "average_precision": np.mean(valid_precisions),
        "average_recall": np.mean(valid_recalls),
        "average_f1": np.mean(valid_f1s),
        "std_f1": np.std(valid_f1s),
        "average_gmean": np.mean(valid_gmeans),
        "std_gmean": np.std(valid_gmeans),
        "average_auc": np.mean(valid_aucs),
        "std_auc": np.std(valid_aucs)
    }
    
    return output_dict
    

In [ ]:
import json
from datetime import datetime

#results_1 = shuffle_data_and_train_and_evaluate(X, y, pipelines_dict["base_mlp"])
#results_2 = shuffle_data_and_train_and_evaluate(X, y, pipelines_dict["smote_mlp"])
#results_3 = shuffle_data_and_train_and_evaluate(X, y, pipelines_dict["adasyn_mlp"])

# iterate through pipelines dict and train the model for each, add results to new dict
results_dict = {}
for name, pipeline in pipelines_dict.items():
    print(f"Training and evaluating pipeline: {name}")
    results = shuffle_data_and_train_and_evaluate(X, y, pipeline)
    results_dict[name] = results
    print("\n" + "="*50 + "\n")
    

cur_date = datetime.now().strftime("%Y%m%d_%H%M%S")
with open("binary_classification_results" + cur_date + ".json", "w") as f:
    json.dump(results_dict, f, indent=4)
    





Training and evaluating pipeline: base_mlp
Run 10/100 completed...
Run 20/100 completed...
Run 30/100 completed...
Run 40/100 completed...
Run 50/100 completed...
Run 60/100 completed...
Run 70/100 completed...
Run 80/100 completed...
Run 90/100 completed...
Run 100/100 completed...
Seeds checked: 100
Average Precision: 0.4357
Average Recall:    0.1513

Average F1:        0.2095
Standard deviation F1: 0.1527
Average G-Mean:    0.3250
standard deviation G-Mean: 0.2119
Average AUC:      0.9059
standard deviation AUC: 0.0709


Training and evaluating pipeline: smote_mlp
Run 10/100 completed...
Run 20/100 completed...
Run 30/100 completed...
Run 40/100 completed...
Run 50/100 completed...
Run 60/100 completed...
Run 70/100 completed...
Run 80/100 completed...
Run 90/100 completed...
Run 100/100 completed...
Seeds checked: 100
Average Precision: 0.2318
Average Recall:    0.6773

Average F1:        0.3417
Standard deviation F1: 0.0634
Average G-Mean:    0.7953
standard deviation G-Mean: 0.07